In [27]:
import pandas as pd
import numpy as np
import re
from pathlib import Path 


In [28]:
# Labels for answer choices the data leaves without one: a bare number ('02') or no value label at all (NaN).
# A scale is identified by its labelled choices, in code order; its entry gives the label of every unlabelled code.
# The same scale gets the same labels in any dataset. Labelled choices are never changed.
LABEL_BANK = {
    ("None", "1 child", "2 children"): {
        3: "3 children",
        4: "4 children",
        5: "5 children",
        6: "6 children",
        7: "7 children",
        8: "8 children",
        9: "9 children",
        10: "10 children",
    },
    ("Yes, they definitely should", "No, they definitely should not"): {
        2: "Yes, they should",
        3: "Yes, they probably should",
        4: "Undecided",
        5: "No, they probably should not",
        6: "No, they should not",
    },
    ("You cannot rely on either of them", "You can definitely rely more on close relatives",
     "You can rely equally on both", "You can definitely rely more on close friends"): {
        2: "You can rely more on close relatives",
        3: "You can probably rely more on close relatives",
        5: "You can probably rely more on close friends",
        6: "You can rely more on close friends",
    },
    ("Close relatives should definitely be the most important", "Both should be equally important",
     "Close friends should definitely be the most important"): {
        2: "Close relatives should be the most important",
        3: "Close relatives should probably be the most important",
        5: "Close friends should probably be the most important",
        6: "Close friends should be the most important",
    },
}

BRACKETS = {"[COUNTRY's]": "{country}'s", "[COUNTRY]": "{country}", "[CNTRY]": "{country}",
            "[EU/ USMCA/ etc.]": "{association}"}                       # the data's placeholders -> the bank's

def fill_placeholders(text, fields, where=""):
    """Text with this country's names in place of placeholders: the bank's {country}, {country}'s, {nationality},
    {parliament}, {association}, and the data's own bracket forms (BRACKETS). A 'the' that starts a sentence is
    capitalised. Prints every placeholder filled (where: e.g. the question id), and any it cannot fill."""
    name = fields["country"]
    values = {"{country}'s": name + ("'" if name.endswith("s") else "'s"), **{f"{{{k}}}": v for k, v in fields.items()}}
    for bracket, brace in BRACKETS.items():
        text = text.replace(bracket, brace)
    for placeholder, value in values.items():
        if placeholder in text:
            print(f"{where:<5} {placeholder:<15} ->  {value!r}")
            text = text.replace(placeholder, value)
    text = re.sub(r"(^|[.?!…]\s+)the ", lambda m: m.group(1) + "The ", text)
    for left in re.findall(r"\{[^}]*\}|\[[^\]]*\]", text):
        print(f"{where:<5} {left:<15} ->  UNFILLED (no value for it)")
    return text

def unlabelled(text):
    return not isinstance(text, str) or not re.search(r"[A-Za-z]", text)   # NaN or no letters ('02')

def scale_key(texts):
    return tuple(t for t in texts if not unlabelled(t))                      # labelled choices in code order

def fill_labels(codes, texts):
    fixes = LABEL_BANK.get(scale_key(texts), {})
    return [fixes.get(c, t) if unlabelled(t) else t for c, t in zip(codes, texts)]


COUNTRY_NOTE = re.compile(r"^([A-Z]{2}(?:\s*,\s*[A-Z]{2})*)\s*:\s*")   # 'NL, US: Employers' -> offered only in NL and US

def offered_options(labels, country, verbose=False):
    """{code: text} of every valid (non-negative) option offered in `country`, in code order. An option marked for
    other countries only is dropped; an option marked for `country` keeps its text without the marker; a country note
    after the text ('Never; PL: 1-2 times a year') gives this country's wording if it names this country, else is cut.
    verbose=True prints every option it drops or changes."""
    out = {}
    for k, t in sorted(labels.items()):
        if k < 0:
            continue
        text = re.sub(r"^-?\d+\.\s*", "", t)
        m = COUNTRY_NOTE.match(text)
        if m:
            if country not in re.split(r"\s*,\s*", m[1]):
                if verbose:
                    print(f"  {int(k):>4}  {text!r}  ->  dropped: not offered in {country}")
                continue
            if verbose:
                print(f"  {int(k):>4}  {text!r}  ->  {text[m.end():]!r}")
            text = text[m.end():]
        own = persona_text(text, country)                              # country notes after the text
        if verbose and own != text:
            print(f"  {int(k):>4}  {text!r}  ->  {own!r}")
        out[int(k)] = own
    return out

def persona_text(label, country):
    """Persona value as this country's respondents saw it: '1. Male' -> 'Male'; 'Once a year; US: Once or twice a year'
    -> 'Once or twice a year' in the US, 'Once a year' elsewhere."""
    text = re.sub(r"^-?\d+\.\s*", "", str(label)).strip()
    kept, ours = [], None
    for part in re.split(r";|(?<![A-Z]{2}),\s*(?=[A-Z]{2}(?:\s*,\s*[A-Z]{2})*\s*:)", text):
        part = part.strip()
        m = COUNTRY_NOTE.match(part)
        if not m:
            kept.append(part)                          # general wording
        elif country in re.split(r"\s*,\s*", m[1]):
            ours = part[m.end():]                      # this country's own wording
    return ours if ours is not None else "; ".join(kept)

In [29]:
data_dir = Path.cwd()/'data/issp2019'
data_fn = data_dir/'raw/ZA7600_v3-0-0.dta'
bank_fn = data_dir/'question_bank.csv'
verified_fn = data_dir/'questions_verified.csv'
country = 'US'
long_fn = data_dir/f"issp2019_{country}_long.parquet"

max_options = 15
countries = pd.read_csv(Path.cwd()/'data/issp_countries.csv', keep_default_na=False).set_index("c_alphan")
fields = {**countries.loc[country, ["country", "nationality", "parliament"]].to_dict(),   # this country's names
          "association": "the United States-Mexico-Canada Agreement (USMCA)"}             # 2023 Q38-Q43: data/issp2023/build_dataset.py

bank = pd.read_csv(bank_fn)

wide = pd.read_stata(data_fn)                                   # as stored; answers shown with GESIS's labels
raw = pd.read_stata(data_fn, convert_categoricals=False)        # NEW: same file, answers as integer codes
labels = pd.io.stata.StataReader(data_fn).variable_labels()     # column -> GESIS label
with pd.io.stata.StataReader(data_fn) as reader:
    label_sets = reader.value_labels()                                   # label-set name -> {code: "N. text"}
    value_labels = {v: label_sets.get(s, {}) for v, s in zip(reader._varlist, reader._lbllist)}   # column -> {code: "N. text"}

if "EDULEVEL" not in raw.columns:                                       # up to 2019 education is DEGREE: same role, its own categories
    wide, raw = wide.rename(columns={"DEGREE": "EDULEVEL"}), raw.rename(columns={"DEGREE": "EDULEVEL"})
    value_labels["EDULEVEL"], labels["EDULEVEL"] = value_labels["DEGREE"], labels["DEGREE"]

wide = wide[wide["c_alphan"] == country]
raw = raw.loc[wide.index]                                       # NEW: same rows as wide

wide["CASEID"] = wide["CASEID"].astype("int64")             # stored as a float; shown as 2.022008e+15 otherwise

persona = ["c_alphan", "SEX", "AGE", "EDULEVEL", "RELIGGRP", "ATTEND", "PARTY_LR", "VOTE_LE", "URBRURAL",
           "MARITAL", "MAINSTAT", "UNION", f"{country}_PRTY", f"{country}_INC", f"{country}_REG", f"{country}_ETHN1"]
questions = [c for c in wide.columns if re.fullmatch(r"v\d+", c)]   # v1, v2, ...: the substantive questions

def all_options(v):
    """{code: text} of every option of question v in this country, in code order: the labelled options offered here
    (offered_options), plus every non-negative code without a label between the labelled ends or given by a respondent
    here (text NaN, filled by LABEL_BANK in the next cell). Codes labelled only for other countries stay out."""
    opts = offered_options(value_labels[v], country)
    given = set(raw.loc[raw[v] >= 0, v].astype(int))                   # codes respondents in this country gave
    labelled = {int(k) for k in value_labels[v] if k >= 0}              # codes with a value label (any country)
    span = set(range(min(opts), max(opts) + 1)) if opts else set()      # every code between the labelled ends
    for c in (span - labelled) | (given - set(opts)):
        opts[c] = float("nan")
    return dict(sorted(opts.items()))

options = {v: all_options(v) for v in questions}                        # v -> {code: text}, every option in this country

wide[questions] = wide[questions].where(raw[questions] >= 0)                          # NEW: negative answers -> NaN (that cell only)

questions = [v for v in questions                                              # the one place questions are kept or dropped:
             if v in set(bank.loc[bank["type"] != "exclude", "v"])            #   in the bank and not marked exclude
             and len(options[v]) <= max_options]                              #   at most max_options offered options

bank = bank[bank["v"].isin(questions)]

# this country's names in place of placeholders, in question texts and option texts
bank["question"] = [fill_placeholders(t, fields, v) for v, t in zip(bank["v"], bank["question"])]
options = {v: {c: fill_placeholders(t, fields, v) if isinstance(t, str) else t for c, t in o.items()} for v, o in options.items()}

wide = wide[["CASEID"] + persona + questions]

wide[questions] = wide[questions].apply(lambda s: s.cat.set_categories(
    [value_labels[s.name].get(k, k) for k in options[s.name]]))         # categories = every option in this country, in code order (an unlabelled code is its own category)

# persona: number prefix removed, this country's wording taken from country notes, negative codes -> NaN
for c in persona[1:]:                                             # c_alphan stay as it is
    vl = value_labels.get(c, {})
    values = {}
    for k in sorted(raw[c].dropna().unique()):
        before = re.sub(r"^-?\d+\.\s*", "", str(vl.get(k, k))).strip()
        if k < 0:
            values[k] = float("nan")
            print(f"{c:<10} {int(k):>6}  {before!r}  ->  NaN  ({int((raw[c] == k).sum())} respondents)")
        else:
            values[k] = persona_text(vl.get(k, k), country)
            if values[k] != before:
                print(f"{c:<10} {int(k):>6}  {before!r}  ->  {values[k]!r}")
    wide[c] = raw[c].map(values)

long = wide.melt(id_vars=["CASEID"]+persona, value_vars=questions, var_name="v", value_name="answer")
long.insert(long.columns.get_loc("v") + 1, "label", long["v"].map(labels))
long["answer_code"] = raw[questions].melt()["value"].astype("Int64").to_numpy()   # CHANGED: code from the raw integers, not the label string
long = long.dropna(subset=["answer"])  # CHANGED: drop the NaN cells (negative or missing answers)

long.head(2)

v21   {country}       ->  'the United States'
v26   {country}       ->  'the United States'
v27   {country}       ->  'the United States'
v29   {country}       ->  'the United States'
v32   {country}       ->  'the United States'
v36   {country}       ->  'the United States'
v37   {country}       ->  'the United States'
v38   {country}       ->  'the United States'
v39   {country}       ->  'the United States'
v40   {country}       ->  'the United States'
v48   {country}       ->  'the United States'
v49   {country}       ->  'the United States'
v50   {country}       ->  'the United States'
v63   {country}'s     ->  "the United States'"
SEX            -9  'No answer'  ->  NaN  (12 respondents)
AGE            -9  'No answer'  ->  NaN  (103 respondents)
AGE            89  '89 years; US: 89 years or more'  ->  '89 years or more'
EDULEVEL       -9  'No answer'  ->  NaN  (7 respondents)
RELIGGRP       -8  'Information insufficient'  ->  NaN  (13 respondents)
RELIGGRP        4  'Other Christ

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,MAINSTAT,UNION,US_PRTY,US_INC,US_REG,US_ETHN1,v,label,answer,answer_code
0,2019008400000001,US,Female,65,Upper secondary (programs that allows entry to...,Catholic,Less frequently than once a year,Center / liberal,Yes,The suburbs or outskirts of a big city,...,In paid work,"No, never, US: No",Democrat,NaN,New England,Italy,v1,Q1a Getting ahead: How important is coming fro...,3. Fairly important,3
1,2019008400000002,US,Male,59,"Lower level tertiary, first stage (also techni...",No religion,Never,Center / liberal,Yes,The suburbs or outskirts of a big city,...,In paid work,"No, never, US: No",Democrat,"170.000 USD or more per year, before taxes",New England,England and Wales,v1,Q1a Getting ahead: How important is coming fro...,3. Fairly important,3


In [30]:
# bank: every option offered in this country, in code order; label fixes shown before -> after
info = bank.set_index("v")
rows = []
for v in questions:
    before = {int(k): re.sub(r"^-?\d+\.\s*", "", t) for k, t in sorted(value_labels[v].items()) if k >= 0}   # as the data has it
    before = dict(sorted((before | {c: float("nan") for c in options[v] if c not in before}).items()))      # + codes without a label
    opts = options[v]                                                                                       # country markers handled
    codes = list(opts)
    after = dict(zip(codes, fill_labels(codes, list(opts.values()))))                                       # unlabelled choices filled
    if any(unlabelled(t) for t in after.values()):                     # no LABEL_BANK entry: the data's numbered form for the whole scale
        after = {c: (t if re.match(rf"{c}\s", t) else f"{c}. {t}") if not unlabelled(t) else f"{c}." for c, t in after.items()}
    rows.append({"v": v,
                 "answer_code": codes,
                 "answer": [value_labels[v].get(k, str(k)) for k in codes],
                 "answer_text": list(after.values())})
    if before == after:
        continue
    print("=" * 100)
    print(f"{v} | {info.loc[v, 'type']} | {info.loc[v, 'data_label']}")
    if any(unlabelled(t) for t in after.values()):
        print("NO ENTRY IN LABEL_BANK FOR THIS SCALE")
    print("-" * 100)
    for c, b in before.items():
        print(f"{c:>4}  {str(b):<45} ->  {after.get(c, f'(dropped: not offered in {country})')}")
print("=" * 100)

bank = bank.merge(pd.DataFrame(rows), on="v")

# long: each person's answer text taken from bank, so the two always agree
text_of = {(r["v"], c): t for _, r in bank.iterrows() for c, t in zip(r["answer_code"], r["answer_text"])}
long["answer_text"] = [text_of[(v, c)] for v, c in zip(long["v"], long["answer_code"])]

# long: the question each answer belongs to, with its type, direction and every option offered
long = long.merge(
    bank[["v", "question", "type", "reversed", "answer_code", "answer_text"]]
        .rename(columns={"answer_code": "option_codes", "answer_text": "option_texts"}),
    on="v", how="left")
long.head(2)

v32 | score | Q10 Do you feel angry about differences in wealth between the rich and the poor?
NO ENTRY IN LABEL_BANK FOR THIS SCALE
----------------------------------------------------------------------------------------------------
   0  Not angry at all                              ->  0. Not angry at all
   1  01                                            ->  1.
   2  02                                            ->  2.
   3  03                                            ->  3.
   4  04                                            ->  4.
   5  05                                            ->  5.
   6  06                                            ->  6.
   7  07                                            ->  7.
   8  08                                            ->  8.
   9  09                                            ->  9.
  10  Extremely angry                               ->  10. Extremely angry
v41 | score | Q13a [TOPBOT] Groups tending towards top+bottom. Where would you put 

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,v,label,answer,answer_code,answer_text,question,type,reversed,option_codes,option_texts
0,2019008400000001,US,Female,65,Upper secondary (programs that allows entry to...,Catholic,Less frequently than once a year,Center / liberal,Yes,The suburbs or outskirts of a big city,...,v1,Q1a Getting ahead: How important is coming fro...,3. Fairly important,3,Fairly important,To begin we have some questions about opportun...,score,True,"[1, 2, 3, 4, 5]","[Essential, Very important, Fairly important, ..."
1,2019008400000002,US,Male,59,"Lower level tertiary, first stage (also techni...",No religion,Never,Center / liberal,Yes,The suburbs or outskirts of a big city,...,v1,Q1a Getting ahead: How important is coming fro...,3. Fairly important,3,Fairly important,To begin we have some questions about opportun...,score,True,"[1, 2, 3, 4, 5]","[Essential, Very important, Fairly important, ..."


In [31]:
long.columns

Index(['CASEID', 'c_alphan', 'SEX', 'AGE', 'EDULEVEL', 'RELIGGRP', 'ATTEND',
       'PARTY_LR', 'VOTE_LE', 'URBRURAL', 'MARITAL', 'MAINSTAT', 'UNION',
       'US_PRTY', 'US_INC', 'US_REG', 'US_ETHN1', 'v', 'label', 'answer',
       'answer_code', 'answer_text', 'question', 'type', 'reversed',
       'option_codes', 'option_texts'],
      dtype='str')

In [32]:
long[long.v=='v10'].answer_text.value_counts()
# long[long.v=='v10'].question.value_counts()

answer_text
Not important at all    689
Not very important      416
Fairly important        399
Very important          195
Essential                69
Name: count, dtype: int64

In [33]:
# for q in long['question'].unique():
#     print(q)
#     print('+'*50)

In [34]:
qid = 'v38'
# qid = 'v46'
long[long['v']==qid].answer.value_counts()

answer
3. Not very strong conflicts    780
2. Strong conflicts             721
1. Very strong conflicts        140
4. There are no conflicts        46
Name: count, dtype: int64

In [35]:
import random

n = 2
long
persona
persona_cols = random.choices(persona, k=n)
long[['v']+persona_cols+['question', 'answer_code', 'option_texts']]


,v,PARTY_LR,UNION,question,answer_code,option_texts
0,v1,Center / liberal,"No, never, US: No",To begin we have some questions about opportun...,3,"[Essential, Very important, Fairly important, ..."
1,v1,Center / liberal,"No, never, US: No",To begin we have some questions about opportun...,3,"[Essential, Very important, Fairly important, ..."
2,v1,Center / liberal,Yes,To begin we have some questions about opportun...,4,"[Essential, Very important, Fairly important, ..."
3,v1,Center / liberal,"No, never, US: No",To begin we have some questions about opportun...,2,"[Essential, Very important, Fairly important, ..."
4,v1,NaN,"No, never, US: No",To begin we have some questions about opportun...,3,"[Essential, Very important, Fairly important, ..."
...,...,...,...,...,...,...
84958,v61,Center / liberal,"No, never, US: No",Most people see themselves as belonging to a p...,3,"[Lower class, Working class, Lower middle clas..."
84959,v61,Center / liberal,"No, never, US: No",Most people see themselves as belonging to a p...,4,"[Lower class, Working class, Lower middle clas..."
84960,v61,NaN,"No, never, US: No",Most people see themselves as belonging to a p...,4,"[Lower class, Working class, Lower middle clas..."
84961,v61,Center / liberal,"No, never, US: No",Most people see themselves as belonging to a p...,4,"[Lower class, Working class, Lower middle clas..."


In [36]:
missing = bank[bank["answer_text"].apply(
    lambda texts: any(not isinstance(t, str) or not re.search(r"[A-Za-z]", t) for t in texts))]
missing[["v", "type", "data_label", "answer_code", "answer_text"]]


,v,type,data_label,answer_code,answer_text
21,v32,score,Q10 Do you feel angry about differences in wea...,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[0. Not angry at all, 1., 2., 3., 4., 5., 6., ..."
30,v41,score,Q13a [TOPBOT] Groups tending towards top+botto...,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[1. Bottom, Lowest, 01, 2., 3., 4., 5., 6., 7...."
31,v42,score,"Q13b Where did the family that you grew up in,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[1. Bottom, Lowest, 01, 2., 3., 4., 5., 6., 7...."
32,v43,score,"Q13c Ahead 10 years from now, where do you thi...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[1. Bottom, Lowest, 01, 2., 3., 4., 5., 6., 7...."


In [37]:
bank[bank['v']=='v33']

,v,source_q,data_label,type,question,codes,options,reversed,note,country_options,country_dropped,answer_code,answer,answer_text
22,v33,Q11a,Q11a Present economic differences between rich...,score,Turning to international differences. To what ...,5 | 4 | 3 | 2 | 1,Strongly disagree | Disagree | Neither agree n...,True,NaN,NaN,NaN,"[1, 2, 3, 4, 5]","[1. Strongly agree, 2. Agree, 3. Neither agree...","[Strongly agree, Agree, Neither agree nor disa..."


In [38]:
long[long['v']=='v60'].answer.value_counts()

Series([], Name: count, dtype: int64)

### Check persona

In [ ]:
bad_persona = {}
for c in persona:
    print(c)
    print(list(long[c].unique()))
    print("+" * 50)
    invalid = [
        v for v in long[c].dropna().unique()
        if re.search(r";|\b[A-Z]{2}(?:\s*,\s*[A-Z]{2})*\s*:|^-?\d+\.\s+|\b(No answer|Don't know|Refused|NAP|NAV|Not available|Can't choose)\b", str(v), re.I)
    ]
    if invalid:
        bad_persona[c] = invalid
if bad_persona:
    raise ValueError(f"Unnormalized persona values found: {bad_persona}")

# Export verified

In [39]:
# ---- export for training: one row per answer, with the person's persona, the question and every option offered
columns = ["CASEID"] + persona + ["v", "label", "question", "type", 
                                  "reversed", "option_codes", "option_texts",
                                  "answer_code", "answer_text"
                                ]
long[columns].to_parquet(long_fn, index=False)
long_fn


PosixPath('/home/dw/github/decision-models-finetuning/data/issp2019/issp2019_US_long.parquet')

In [40]:
bank['options_old'] = bank['options']
bank[['v', 'data_label', 'question', 'options_old', 'answer_code', 'answer_text', 'type', 'reversed']].to_csv(verified_fn,index=False)
